# GHSL data preparation — Milan raster, resampled points, Sentinel-2 bands

Packaged from three scripts written and run in sequence this session
(`extract_ghsl_milan.py`, `resample_ghsl_points.py`, `extract_ghsl_s2_bands.py`).
Each cell below is that script's exact tested source; the two cross-script
imports (`from extract_ghsl_milan import ...`, `from resample_ghsl_points
import ...`) are dropped since the earlier cell already defines those names
in this notebook's namespace once run top to bottom.

**This is a record of work already done, not meant for casual re-execution.**
`resample_ghsl_points.py`'s cell took ~619s and calls Earth Engine; the S2
extraction cell also calls Earth Engine and can take minutes per composite.
Re-running is safe (idempotent — each cell overwrites its own output files),
but there is no reason to unless an input changed.

## 1. Clip, reproject and reclassify GHSL for Milan; relabel the CLMS sample points

`extract_ghsl_milan.py` — clips the global GHS-BUILT-S tile to the Milan AOI, reprojects onto the CLMS grid, reclassifies into the 7 IMD classes, and (as a first pass, superseded for training use by cell 2's independent resample) relabels the existing CLMS sample points with GHSL/GHSL_class.

In [ ]:
"""Extract GHSL built-up surface (GHS-BUILT-S, 2018) at the Milan sample points.

A same-location alternative label to the CLMS-derived IMD, so a model trained
against GHSL can be compared with the CLMS run without re-sampling new points.
No Earth Engine needed -- everything here is local raster/vector work.

Three outputs:
  data/GHSL_2018_Milan_UTM32N.tif        GHSL clipped out of the global tile
                                          and reprojected onto the exact pixel
                                          grid of data/CLMS_2018_Milan_UTM32N.tif
                                          (EPSG:32632, 10 m) -- so the two are
                                          diff-able cell for cell.
  data/GHSL_2018_Milan_UTM32N_class.tif  The above reclassified into the 7 IMD
                                          classes (0 / 1-20 / 21-40 / 41-60 /
                                          61-80 / 81-99 / 100), done once over
                                          the whole raster before any point
                                          sampling -- so GHSL_class always
                                          traces back to a raster artifact you
                                          can open and check, not a value
                                          recomputed ad hoc per point.
  outputs_sampling/sample_points_all_GHSL.gpkg
                                          A drop-in replacement for
                                          outputs_sampling/sample_points_all.gpkg:
                                          same 3500 points, same A00-A63
                                          AlphaEarth columns, but GHSL/GHSL_class
                                          in place of IMD/IMD_class -- never
                                          both labels in one file, so notebook
                                          01c's ALL_POINTS_PATH can point at
                                          either interchangeably.
  samples_S2_<tag>/sample_points_all_S2_GHSL.gpkg
                                          Same drop-in swap for every Sentinel-2
                                          baseline table notebook 00 has built --
                                          median, stack, percentile_pXXpYY..., one
                                          per samples_S2_<tag>/ directory found.
                                          Same 3500 points, same bands (unaffected
                                          by the label, so nothing is re-extracted
                                          from GEE), GHSL/GHSL_class in place of
                                          IMD/IMD_class. Feeds notebook 01d --
                                          re-run this script after notebook 00
                                          finishes a new mode and it picks up the
                                          new samples_S2_<tag>/ automatically.

Run: python extract_ghsl_milan.py
"""
import glob
import os

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import Resampling, reproject, transform_bounds
from rasterio.windows import Window

RAW_GHSL_PATH = (r'C:\Users\user\Downloads'
                  r'\GHS_BUILT_S_E2018_GLOBE_R2023A_54009_10_V1_0_R4_C19'
                  r'\GHS_BUILT_S_E2018_GLOBE_R2023A_54009_10_V1_0_R4_C19.tif')
REF_RASTER_PATH  = 'data/CLMS_2018_Milan_UTM32N.tif'
OUT_RASTER_PATH  = 'data/GHSL_2018_Milan_UTM32N.tif'
CLASS_RASTER_PATH = 'data/GHSL_2018_Milan_UTM32N_class.tif'

AEF_SRC_POINTS_PATH = 'data/sample_points/sample_points_all_CLMS_Milan.gpkg'
AEF_OUT_POINTS_PATH = 'data/sample_points/sample_points_all_GHSL_Milan.gpkg'

S2_SAMPLE_GLOB = 'output/milan/clms/*/sample_points_all_S2.gpkg'

NODATA       = 255   # GHS-BUILT-S convention: 0-100 = % built, 255 = no data
CLASS_NODATA = 255   # no valid class is 255, so it doubles as the class marker

CLASS_LABELS = ['C0 (0%)', 'C1 (1-20%)', 'C2 (21-40%)', 'C3 (41-60%)',
                'C4 (61-80%)', 'C5 (81-99%)', 'C6 (100%)']


def reclass_imd_7(arr):
    """Continuous 0-100 density -> the 7 IMD classes (matches notebook 02's
    reclass_imd_7, so GHSL_class lines up with the existing IMD_class)."""
    out = np.full(arr.shape, -1, dtype=np.int8)
    out[arr == 0]                 = 0
    out[(arr > 0)  & (arr <= 20)] = 1
    out[(arr > 20) & (arr <= 40)] = 2
    out[(arr > 40) & (arr <= 60)] = 3
    out[(arr > 60) & (arr <= 80)] = 4
    out[(arr > 80) & (arr < 100)] = 5
    out[arr == 100]               = 6
    return out


def clip_and_align_ghsl():
    """Clip the raw 100000x100000 px global GHSL tile down to the Milan AOI
    and reproject it onto the CLMS reference raster's exact grid."""
    with rasterio.open(REF_RASTER_PATH) as ref:
        ref_crs, ref_transform = ref.crs, ref.transform
        ref_shape = (ref.height, ref.width)
        ref_bounds = ref.bounds

    with rasterio.open(RAW_GHSL_PATH) as src:
        # Window-read only the Milan area -- reprojecting the whole global
        # tile would mean reading ~10 billion pixels.
        moll_bounds = transform_bounds(ref_crs, src.crs, *ref_bounds, densify_pts=21)
        window = rasterio.windows.from_bounds(*moll_bounds, transform=src.transform)
        pad = 20  # px, absorbs reprojection edge effects
        window = Window(window.col_off - pad, window.row_off - pad,
                         window.width + 2 * pad, window.height + 2 * pad)
        src_arr       = src.read(1, window=window)
        src_transform = src.window_transform(window)
        src_crs       = src.crs
        src_nodata    = src.nodata

    dst_arr = np.full(ref_shape, NODATA, dtype=np.uint8)
    reproject(
        source=src_arr, destination=dst_arr,
        src_transform=src_transform, src_crs=src_crs, src_nodata=src_nodata,
        dst_transform=ref_transform, dst_crs=ref_crs, dst_nodata=NODATA,
        resampling=Resampling.bilinear,
    )

    os.makedirs(os.path.dirname(OUT_RASTER_PATH), exist_ok=True)
    profile = dict(driver='GTiff', height=ref_shape[0], width=ref_shape[1],
                    count=1, dtype='uint8', crs=ref_crs, transform=ref_transform,
                    nodata=NODATA, compress='deflate')
    with rasterio.open(OUT_RASTER_PATH, 'w', **profile) as dst:
        dst.write(dst_arr, 1)

    valid = dst_arr != NODATA
    print(f'GHSL aligned to the CLMS grid -> {OUT_RASTER_PATH}')
    print(f'  shape: {dst_arr.shape} | valid: {valid.sum()}/{dst_arr.size} '
          f'({100 * valid.mean():.1f}%) | range: '
          f'{dst_arr[valid].min()}-{dst_arr[valid].max()}%')

    # Reclassify the whole raster -- initially, before any point sampling --
    # into the 7 IMD classes. reclass_imd_7 fills unmatched cells with -1,
    # and int8(-1) casts to uint8(255) automatically, which is CLASS_NODATA:
    # the 255 GHSL nodata pixels fall out of every class test and land there
    # too, so no separate masking step is needed.
    cls_arr = reclass_imd_7(dst_arr).astype(np.uint8)
    cls_profile = dict(driver='GTiff', height=ref_shape[0], width=ref_shape[1],
                        count=1, dtype='uint8', crs=ref_crs, transform=ref_transform,
                        nodata=CLASS_NODATA, compress='deflate')
    with rasterio.open(CLASS_RASTER_PATH, 'w', **cls_profile) as dst:
        dst.write(cls_arr, 1)

    print(f'GHSL reclassified -> {CLASS_RASTER_PATH}')
    for c, label in enumerate(CLASS_LABELS):
        n = int((cls_arr == c).sum())
        print(f'  {label}: {n:>10,} px  ({100 * n / valid.sum():5.1f}% of valid area)')


def _sample_ghsl_at(coords_wgs84):
    """Value + class from the two rasters built in clip_and_align_ghsl(), at
    a list of (lon, lat) points -- shared by both relabelling calls below."""
    gdf_pts = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(*zip(*coords_wgs84)), crs='EPSG:4326'
    ).to_crs('EPSG:32632')
    coords = [(geom.x, geom.y) for geom in gdf_pts.geometry]

    with rasterio.open(OUT_RASTER_PATH) as src:
        vals = np.array([v[0] for v in src.sample(coords)], dtype=float)
        vals[vals == src.nodata] = np.nan

    # Read the class off the raster reclassified in clip_and_align_ghsl(),
    # rather than recomputing it from vals -- one reclassification, done once
    # on the raster, is the source of truth for both the raster and the points.
    with rasterio.open(CLASS_RASTER_PATH) as src:
        cls = np.array([v[0] for v in src.sample(coords)], dtype=int)

    return vals, cls


def relabel_points(src_path, out_path):
    """Read a sample-points gpkg carrying IMD/IMD_class, and write a copy at
    the same 3500 locations with GHSL/GHSL_class standing in for them --
    never both labels in one file. Every other column (AlphaEarth bands or
    Sentinel-2 bands) passes through untouched, since predictors don't depend
    on which label they're being compared against."""
    gdf_src = gpd.read_file(src_path)
    assert len(gdf_src) == 3500, f'Expected 3500 source points, got {len(gdf_src)}'
    keep_cols = [c for c in gdf_src.columns if c not in ('IMD', 'IMD_class', 'geometry')]

    gdf_wgs84 = gdf_src if gdf_src.crs.to_epsg() == 4326 else gdf_src.to_crs('EPSG:4326')
    coords = [(geom.x, geom.y) for geom in gdf_wgs84.geometry]
    vals, cls = _sample_ghsl_at(coords)

    missing = np.isnan(vals)
    if missing.any():
        print(f'{int(missing.sum())} of {len(vals)} points have no GHSL value '
              '(fell outside the downloaded tile or on a masked pixel).')

    gdf_out = gdf_src[keep_cols + ['geometry']].copy()
    gdf_out['GHSL']       = vals
    gdf_out['GHSL_class'] = cls

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    gdf_out.to_file(out_path, driver='GPKG')
    print(f'Wrote {len(gdf_out)} points -> {out_path}')
    print(f'Columns: {list(gdf_out.columns)}')

    # Sanity check only -- IMD never enters the output file.
    both = ~missing
    diff = vals[both] - gdf_src['IMD'].values[both]
    print(f'[sanity check, not written] GHSL vs CLMS IMD at the same '
          f'{both.sum()} points (GHSL - IMD): mean={diff.mean():+.2f}pp  '
          f'std={diff.std():.2f}pp\n')


if __name__ == '__main__':
    clip_and_align_ghsl()
    relabel_points(AEF_SRC_POINTS_PATH, AEF_OUT_POINTS_PATH)

    s2_tables = sorted(glob.glob(S2_SAMPLE_GLOB))
    if not s2_tables:
        print(f'No Sentinel-2 tables found ({S2_SAMPLE_GLOB}) -- run notebook '
              '00 for at least one COMPOSITE_METHOD before this script can '
              'relabel it.')
    for src_path in s2_tables:
        tag = os.path.basename(os.path.dirname(src_path))          # 'median', 'stack', ...
        out_path = src_path.replace('sample_points_all_S2.gpkg',
                                     'sample_points_all_S2_GHSL.gpkg')
        print(f'-- {tag} --')
        relabel_points(src_path, out_path)

## 2. Matej's-method resampling: an independent stratified draw from GHSL's own classification

`resample_ghsl_points.py` — 500 points per class (3 500 total) drawn directly from the GHSL 7-class raster, validated with an Average Nearest Neighbour index, then AlphaEarth and Sentinel-2 (median composite) features extracted at those points via Earth Engine.

In [ ]:
# extract_ghsl_milan's cell above already defines CLASS_LABELS, CLASS_RASTER_PATH, OUT_RASTER_PATH in this notebook's namespace.
"""Resample the Milan GHSL training/test points using Matej Zgela's method
(reference/Zgela_LCZ-UHI-GEO_Report.pdf, Section 1a), instead of reusing the
CLMS point locations.

Why: extract_ghsl_milan.py's relabel_points() reused the CLMS-derived 3500
points and just read GHSL off them. Since CLMS and GHSL disagree pixel to
pixel, that gave a skewed class mix (e.g. 39 points at class 6 instead of
500). Matej's own method for the Vietnam transfer -- "random stratified
sampling of 500 points per class from the GHS-BUILT-S dataset" -- draws points
directly from GHSL's own classification instead, so the GHSL run is balanced
on its own terms, the same way the CLMS run is balanced on its.

Steps (mirrors notebook 02's Cell 3 "Sample 500/class stratified", applied to
Milan's own GHSL raster rather than a new city):
  1. Draw 500 points/class from data/GHSL_2018_Milan_UTM32N_class.tif (built by
     extract_ghsl_milan.py) -- new locations, not the CLMS ones.
  2. Run the same validation checks Matej reports for point selection: the
     Average Nearest Neighbour (ANN) spatial-randomness index per class, and
     per-class value-distribution stats.
  3. Extract AlphaEarth embeddings (A00-A63) and Sentinel-2 bands (B2-B12,
     same 30-date median composite as samples_S2_median) at these new points
     via Earth Engine, chunked as in notebook 00/02.
  4. Overwrite outputs_sampling/sample_points_all_GHSL.gpkg and
     samples_S2_median/sample_points_all_S2_GHSL.gpkg with the new points --
     both share the same new geometries, matching how the CLMS run's 3500
     points are shared between the AEF and S2 tables.

Needs a cached Earth Engine credential (same one 00_/01c/01d already use) --
no ee.Authenticate() prompt if `earthengine authenticate` has already run.

Run: python resample_ghsl_points.py
"""
import json
import os
import time

import ee
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from scipy.spatial import cKDTree

from s2_utils import build_composite, water_mask

GEE_PROJECT   = 'impervious-s2'
N_CLASSES     = 7
SAMPLES_PER_CLASS = 500
# Some drawn points land on water and get masked out of both the AlphaEarth
# and Sentinel-2 extractions (observed ~1.2% worst-case, class 0). Draw extra
# per class up front and trim back to exactly 500 after extraction, rather
# than shipping a shortfall -- Matej's design is exactly 500/class, and so is
# every downstream notebook's `assert len(gdf_all) == 3500`.
OVERSAMPLE_PER_CLASS = 520
RANDOM_STATE  = 42

AEF_OUT_PATH  = 'data/sample_points/sample_points_all_GHSL_Milan.gpkg'
S2_SAMPLE_DIR = 'output/milan/clms/median'
S2_META_PATH  = f'{S2_SAMPLE_DIR}/s2_extraction_metadata.json'
S2_OUT_PATH   = f'{S2_SAMPLE_DIR}/sample_points_all_S2_GHSL.gpkg'
QC_FIG_PATH   = 'output/milan/ghsl/ghsl_resample_qc.png'

CHUNK_SIZE = 500   # points per getInfo batch, matches notebook 00 Cell 8


def sample_ghsl_points_matej():
    """OVERSAMPLE_PER_CLASS points/class drawn directly from the GHSL
    classification -- new locations, independent of the CLMS points. Extra
    points beyond the 500/class Matej uses are trimmed off in main() after
    extraction drops whichever ones land on water."""
    with rasterio.open(CLASS_RASTER_PATH) as src:
        cls_arr   = src.read(1)
        transform = src.transform
        crs       = src.crs
    with rasterio.open(OUT_RASTER_PATH) as src:
        val_arr = src.read(1)

    rng = np.random.RandomState(RANDOM_STATE)
    rows, cols, classes = [], [], []
    for c in range(N_CLASSES):
        ys, xs = np.where(cls_arr == c)
        if len(ys) < OVERSAMPLE_PER_CLASS:
            raise RuntimeError(f'Class {c} has only {len(ys)} px, need '
                                f'{OVERSAMPLE_PER_CLASS}.')
        idx = rng.choice(len(ys), OVERSAMPLE_PER_CLASS, replace=False)
        rows.extend(ys[idx]); cols.extend(xs[idx]); classes.extend([c] * OVERSAMPLE_PER_CLASS)

    rows, cols, classes = np.array(rows), np.array(cols), np.array(classes)
    xs_utm, ys_utm = rasterio.transform.xy(transform, rows, cols)  # pixel centres
    vals = val_arr[rows, cols].astype(float)

    gdf_utm = gpd.GeoDataFrame(
        {'GHSL': vals, 'GHSL_class': classes},
        geometry=gpd.points_from_xy(xs_utm, ys_utm), crs=crs)
    gdf = gdf_utm.to_crs('EPSG:4326').reset_index(drop=True)
    print(f'Sampled {len(gdf)} candidate points ({OVERSAMPLE_PER_CLASS}/class x '
          f'{N_CLASSES} classes -- a buffer over the {SAMPLES_PER_CLASS}/class '
          'target), Matej\'s method, from GHSL\'s own classification.')
    return gdf


def ann_report(coords_utm, classes, aoi_area_m2):
    """Average Nearest Neighbour index per class (Clark & Evans 1954, the
    formula Matej's report cites via the ArcGIS ANN documentation):
      expected mean NN distance under CSR = 0.5 * sqrt(A / n)
      ANN = observed mean NN distance / expected
    ANN << 1 -> clustered, ANN ~= 1 -> random, ANN >> 1 -> dispersed."""
    rows = []
    for c in range(N_CLASSES):
        pts = coords_utm[classes == c]
        tree = cKDTree(pts)
        d, _ = tree.query(pts, k=2)          # k=2: nearest OTHER point (k=1 is itself)
        observed = d[:, 1].mean()
        n = len(pts)
        expected = 0.5 * np.sqrt(aoi_area_m2 / n)
        rows.append({'class': CLASS_LABELS[c], 'n': n,
                      'observed_NN_m': observed, 'expected_NN_m': expected,
                      'ANN': observed / expected})
    df = pd.DataFrame(rows)
    print('\nSpatial randomness (Average Nearest Neighbour index per class):')
    print('  ANN ~= 1 random, < 1 clustered, > 1 dispersed '
          '(Matej reports 1.02 down to 0.75 for Milan/CLMS)')
    print(df.to_string(index=False, float_format=lambda v: f'{v:,.2f}'))
    return df


def value_distribution_report(gdf):
    rows = []
    for c in range(N_CLASSES):
        v = gdf.loc[gdf['GHSL_class'] == c, 'GHSL']
        rows.append({'class': CLASS_LABELS[c], 'n': len(v),
                      'min': v.min(), 'q1': v.quantile(.25), 'mean': v.mean(),
                      'median': v.median(), 'q3': v.quantile(.75), 'max': v.max(),
                      'std': v.std()})
    df = pd.DataFrame(rows)
    print('\nWithin-class value distribution (GHSL %, matches Matej\'s Figure 2 check):')
    print(df.to_string(index=False, float_format=lambda v: f'{v:,.2f}'))
    single_value = df[(df['min'] == df['max'])]['class'].tolist()
    if single_value:
        print(f'  Note: {single_value} are single-valued (0 or 100), same as '
              'Matej reports for classes 0 and 6 -- expected, not a defect.')
    return df


def plot_qc_figure(gdf, ann_df, dist_df):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    colors = plt.cm.viridis(np.linspace(0, 1, N_CLASSES))
    for c in range(N_CLASSES):
        sub = gdf[gdf['GHSL_class'] == c]
        axes[0].scatter(sub.geometry.x, sub.geometry.y, s=4, color=colors[c],
                         label=CLASS_LABELS[c], alpha=.6)
    axes[0].set_title('Resampled points by class'); axes[0].legend(fontsize=6, markerscale=2)
    axes[0].set_xlabel('lon'); axes[0].set_ylabel('lat')

    axes[1].bar(range(N_CLASSES), ann_df['ANN'], color=colors)
    axes[1].axhline(1, color='k', ls='--', lw=1)
    axes[1].set_xticks(range(N_CLASSES)); axes[1].set_xticklabels(range(N_CLASSES))
    axes[1].set_xlabel('class'); axes[1].set_ylabel('ANN index')
    axes[1].set_title('Spatial randomness per class')

    data = [gdf.loc[gdf['GHSL_class'] == c, 'GHSL'].values for c in range(N_CLASSES)]
    axes[2].boxplot(data, tick_labels=range(N_CLASSES))
    axes[2].set_xlabel('class'); axes[2].set_ylabel('GHSL (%)')
    axes[2].set_title('Within-class value spread')

    plt.tight_layout()
    os.makedirs(os.path.dirname(QC_FIG_PATH), exist_ok=True)
    plt.savefig(QC_FIG_PATH, dpi=130, bbox_inches='tight')
    plt.close(fig)
    print(f'\nQC figure -> {QC_FIG_PATH}')


def _sample_image_at_points(image, gdf, scale, projection, tile_scale, wide):
    """Chunked sampleRegions -> DataFrame, matching notebook 00 Cell 8 / 02 Cell 3.

    properties=['pid'] only: that arg copies INPUT feature properties onto the
    output, it does not select which image bands come back -- passing band
    names there (tried first) makes sampleRegions return pid alone and drop
    every band silently. Band columns are added automatically, one per band."""
    chunk_size = CHUNK_SIZE // 2 if wide else CHUNK_SIZE
    records = []
    n_chunks = int(np.ceil(len(gdf) / chunk_size))
    for ci in range(n_chunks):
        chunk = gdf.iloc[ci * chunk_size:(ci + 1) * chunk_size]
        feats = [ee.Feature(ee.Geometry.Point([row.geometry.x, row.geometry.y]),
                             {'pid': int(row['pid'])})
                  for _, row in chunk.iterrows()]
        sampled = image.sampleRegions(
            collection=ee.FeatureCollection(feats), properties=['pid'],
            scale=scale, projection=projection, tileScale=tile_scale, geometries=False)
        records.extend([f['properties'] for f in sampled.getInfo()['features']])
        print(f'  chunk {ci + 1}/{n_chunks}: {len(records)} points returned so far')
    return pd.DataFrame(records)


def extract_aef_at_points(gdf):
    """AlphaEarth embeddings (A00-A63) at the new points -- mirrors notebook
    01_/02_'s AEF image construction."""
    aoi = ee.FeatureCollection(f'projects/{GEE_PROJECT}/assets/milano_aoi')
    aoi_geom = aoi.geometry()
    non_water = water_mask(aoi_geom)

    start = ee.Date.fromYMD(2018, 1, 1)
    aef_col = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
               .filter(ee.Filter.date(start, start.advance(1, 'year')))
               .filter(ee.Filter.bounds(aoi_geom)))
    aef_proj = aef_col.first().projection()
    aef_img = (aef_col.mosaic().setDefaultProjection(aef_proj)
               .clip(aoi_geom).updateMask(non_water))
    band_names = aef_img.bandNames().getInfo()
    print(f'\nExtracting AlphaEarth ({len(band_names)} bands) at {len(gdf)} points...')

    gdf_pid = gdf.copy(); gdf_pid['pid'] = np.arange(len(gdf_pid))
    df = _sample_image_at_points(aef_img, gdf_pid, scale=10,
                                  projection=aef_proj, tile_scale=4, wide=False)
    return _attach_bands(gdf_pid, df, band_names, 'AlphaEarth')


def extract_s2_at_points(gdf):
    """Sentinel-2 median composite (B2-B12) at the new points, using the SAME
    30 dates as samples_S2_median -- so this predictor table is comparable to
    the CLMS-labelled one, differing only in which points/labels are used."""
    with open(S2_META_PATH) as f:
        meta = json.load(f)
    selected_dates = meta['selected_dates']

    aoi = ee.FeatureCollection(f'projects/{GEE_PROJECT}/assets/milano_aoi')
    aoi_geom = aoi.geometry()
    non_water = water_mask(aoi_geom)
    s2_img, s2_proj, band_names = build_composite(
        meta['collection'], selected_dates, aoi_geom, non_water, method='median')
    print(f'\nExtracting Sentinel-2 ({len(band_names)} bands, {len(selected_dates)} dates) '
          f'at {len(gdf)} points...')

    gdf_pid = gdf.copy(); gdf_pid['pid'] = np.arange(len(gdf_pid))
    wide = len(selected_dates) >= 10
    df = _sample_image_at_points(s2_img, gdf_pid, scale=10,
                                  projection=s2_proj,
                                  tile_scale=8 if wide else 4, wide=wide)
    return _attach_bands(gdf_pid, df, band_names, 'Sentinel-2')


def _attach_bands(gdf_pid, df, band_names, label):
    """Reindex the (possibly short -- masked points just don't come back)
    sampleRegions result onto pid 0..N-1, so missing points show up as NaN
    rows instead of silently shifting every later row up."""
    merged = df.set_index('pid').reindex(range(len(gdf_pid)))
    missing = merged[band_names].isna().any(axis=1)
    if missing.any():
        print(f'  {int(missing.sum())} of {len(gdf_pid)} points have no {label} value '
              '(masked/water).')
    out = gdf_pid.copy()
    out[band_names] = merged[band_names].values.astype(np.float32)
    return out, missing.values


def _trim_to_exactly(gdf, keep_mask, per_class):
    """Within each class, keep the first `per_class` surviving rows in their
    original (already-random) draw order, and drop the rest of the buffer."""
    gdf = gdf.copy()
    gdf['_survives'] = keep_mask
    keep_idx = []
    for c in range(N_CLASSES):
        cls_idx = gdf.index[(gdf['GHSL_class'] == c) & gdf['_survives']]
        if len(cls_idx) < per_class:
            raise RuntimeError(
                f'Class {c}: only {len(cls_idx)} surviving points after '
                f'extraction, need {per_class}. Raise OVERSAMPLE_PER_CLASS '
                'and re-run.')
        keep_idx.extend(cls_idx[:per_class])
    return sorted(keep_idx)


def main():
    ee.Initialize(project=GEE_PROJECT)

    gdf = sample_ghsl_points_matej()

    gdf_aef, missing_aef = extract_aef_at_points(gdf)
    gdf_s2,  missing_s2  = extract_s2_at_points(gdf)

    # Keep the AEF and S2 tables on the SAME point set -- matching how the
    # CLMS run's 3500 points are shared identically between the two tracks.
    # A point masked in either extraction is unusable in both.
    survives = ~(missing_aef | missing_s2)
    n_masked = (~survives).sum()
    if n_masked:
        print(f'\n{int(n_masked)} of {len(gdf)} candidate point(s) masked in '
              'AlphaEarth and/or Sentinel-2 (water) -- excluded before trimming.')

    keep_idx = _trim_to_exactly(gdf, survives, SAMPLES_PER_CLASS)
    gdf     = gdf.loc[keep_idx].reset_index(drop=True)
    gdf_aef = gdf_aef.loc[keep_idx].drop(columns=['pid']).reset_index(drop=True)
    gdf_s2  = gdf_s2.loc[keep_idx].drop(columns=['pid']).reset_index(drop=True)
    assert len(gdf) == SAMPLES_PER_CLASS * N_CLASSES, len(gdf)
    assert (gdf['GHSL_class'].value_counts() == SAMPLES_PER_CLASS).all()
    print(f'\nTrimmed to exactly {len(gdf)} points ({SAMPLES_PER_CLASS}/class).')

    # QC tests (Matej's method) run on the FINAL exactly-500/class set only.
    gdf_utm = gdf.to_crs('EPSG:32632')
    coords_utm = np.column_stack([gdf_utm.geometry.x, gdf_utm.geometry.y])
    aoi = ee.FeatureCollection(f'projects/{GEE_PROJECT}/assets/milano_aoi')
    aoi_area_m2 = aoi.geometry().area(1).getInfo()

    ann_df  = ann_report(coords_utm, gdf['GHSL_class'].values, aoi_area_m2)
    dist_df = value_distribution_report(gdf)
    plot_qc_figure(gdf, ann_df, dist_df)

    os.makedirs(os.path.dirname(AEF_OUT_PATH), exist_ok=True)
    gdf_aef.to_file(AEF_OUT_PATH, driver='GPKG')
    print(f'\nWrote {len(gdf_aef)} points -> {AEF_OUT_PATH}')

    os.makedirs(os.path.dirname(S2_OUT_PATH), exist_ok=True)
    gdf_s2.to_file(S2_OUT_PATH, driver='GPKG')
    print(f'Wrote {len(gdf_s2)} points -> {S2_OUT_PATH}')


if __name__ == '__main__':
    t0 = time.time()
    main()
    print(f'\nDone in {time.time() - t0:.0f}s')

## 3. Extract Sentinel-2 bands (stack / percentile composites) at the fixed GHSL points

`extract_ghsl_s2_bands.py` — the median-composite bands come from cell 2; this rebuilds the stack and percentile composites from each city's `s2_extraction_metadata.json` and samples them at the same fixed points.

In [ ]:
# resample_ghsl_points's cell above already defines GEE_PROJECT and _sample_image_at_points in this notebook's namespace.
"""Extract Sentinel-2 bands at the fixed GHSL points, for every composite
method notebook 00 has built (median/stack/percentile), so 01d can model any
of them against GHSL.

The GHSL point set itself is NOT re-sampled here -- it's the one Matej's-method
run already fixed in outputs_sampling/sample_points_all_GHSL.gpkg (3500 points,
500/class, already verified to survive both AlphaEarth and Sentinel-2 masking
under the median composite). This script reuses those exact geometries and
just pulls a different composite's bands at them, mirroring how notebook 00
itself reuses the same 3500 CLMS points across composite methods -- only the
extracted band values change per method, never the points.

For each samples_S2_<tag>/ that notebook 00 has populated (has
sample_points_all_S2.gpkg + s2_extraction_metadata.json) but that has no
sample_points_all_S2_GHSL.gpkg yet, this builds that tag's composite and
extracts it at the fixed GHSL points.

Run: python extract_ghsl_s2_bands.py
"""
import glob
import json
import os

import ee
import geopandas as gpd

from s2_utils import build_composite, water_mask

GHSL_POINTS_PATH = 'data/sample_points/sample_points_all_GHSL_Milan.gpkg'
S2_BANDS_N = 10   # from s2_utils.S2_BANDS -- used for the same "wide" test as notebook 00


def load_fixed_ghsl_points():
    gdf = gpd.read_file(GHSL_POINTS_PATH)[['GHSL', 'GHSL_class', 'geometry']]
    print(f'Loaded {len(gdf)} fixed GHSL points from {GHSL_POINTS_PATH}')
    return gdf


def extract_for_tag(sample_dir, gdf_ghsl):
    meta_path = f'{sample_dir}/s2_extraction_metadata.json'
    out_path  = f'{sample_dir}/sample_points_all_S2_GHSL.gpkg'
    with open(meta_path) as f:
        meta = json.load(f)

    aoi = ee.FeatureCollection(f'projects/{GEE_PROJECT}/assets/milano_aoi')
    aoi_geom = aoi.geometry()
    non_water = water_mask(aoi_geom)
    s2_img, s2_proj, band_names = build_composite(
        meta['collection'], meta['selected_dates'], aoi_geom, non_water,
        method=meta['method'],
        percentiles=tuple(meta['percentiles']) if meta.get('percentiles') else (25, 50, 75))

    assert band_names == meta['bands'], (
        f'{sample_dir}: rebuilt composite bands do not match metadata -- '
        's2_utils.build_composite has changed since notebook 00 ran.')

    # Same "wide" rule as notebook 00 Cell 8.
    wide = len(band_names) > S2_BANDS_N or len(meta['selected_dates']) >= 10
    print(f'\n-- {sample_dir} (method={meta["method"]}, '
          f'{len(meta["selected_dates"])} dates, {len(band_names)} bands) --')

    gdf_pid = gdf_ghsl.copy()
    gdf_pid['pid'] = range(len(gdf_pid))
    df = _sample_image_at_points(s2_img, gdf_pid, scale=10, projection=s2_proj,
                                  tile_scale=8 if wide else 4, wide=wide)

    merged = df.set_index('pid').reindex(range(len(gdf_pid)))
    missing = merged[band_names].isna().any(axis=1)
    if missing.any():
        raise RuntimeError(
            f'{sample_dir}: {int(missing.sum())} of {len(gdf_pid)} fixed GHSL '
            'points have no value under this composite -- unexpected, since '
            'these points already survived the median composite + AlphaEarth '
            'water mask. Inspect before proceeding.')

    gdf_out = gdf_ghsl.copy()
    gdf_out[band_names] = merged[band_names].values

    gdf_out.to_file(out_path, driver='GPKG')
    print(f'Wrote {len(gdf_out)} points -> {out_path}')


def main():
    ee.Initialize(project=GEE_PROJECT)
    gdf_ghsl = load_fixed_ghsl_points()

    candidates = sorted(glob.glob('output/milan/clms/*/sample_points_all_S2.gpkg'))
    if not candidates:
        print('No output/milan/clms/*/sample_points_all_S2.gpkg found -- run notebook 00 first.')
        return

    for src in candidates:
        sample_dir = os.path.dirname(src)
        out_path = f'{sample_dir}/sample_points_all_S2_GHSL.gpkg'
        if os.path.exists(out_path):
            print(f'-- {sample_dir}: {out_path} already exists, skipping.')
            continue
        extract_for_tag(sample_dir, gdf_ghsl)


if __name__ == '__main__':
    main()